### **Authors:**

Juan Navarro - s1097545

*Add your names + s number when possible*


## Part 1: Loading the data

**Load the train, dev and test sets into pandas DataFrames.**

**What we do:**
1. Read the three CSV files (train, dev and test).
2. Check the shape and columns of each set (snippet ID, 'text', 'author').
3. Fix the text: contractions appear with a double quote instead of an apostrophe (e.g. wasn"t), so we restore the apostrophe (wasn't).
4. Check the number of snippets per author.

**Output:** three cleaned DataFrames: 'train', 'dev' and 'test'.

In [11]:
%pip install pandas numpy spacy scikit-learn
!python -m spacy download en_core_web_sm

Note: you may need to restart the kernel to use updated packages.
  Using cached en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [12]:
# Imports
import re
import pandas as pd
import numpy as np
import spacy
from collections import Counter
from sklearn.preprocessing import StandardScaler



In [13]:
#1 Import the csv files as pandas dataframes
train = pd.read_csv('data/pan2627_train_data.csv', index_col=0).rename_axis('id')
dev   = pd.read_csv('data/pan2627_dev_data.csv',   index_col=0).rename_axis('id')
test  = pd.read_csv('data/pan2627_test_data.csv',  index_col=0).rename_axis('id')

#2 Check shapes, columns and author counts
for name, df in [('train', train), ('dev', dev), ('test', test)]:
    print(f'{name}: {df.shape[0]} snippets, columns: {list(df.columns)}')

counts = train['author'].value_counts().sort_index()
print()
print('Number of different authors in train:', train['author'].nunique())
print(f'Fewest snippets for one author: {counts.min()}, most: {counts.max()}')
print()
print('Snippets per author (train):')
print(counts.to_frame('snippets'))

train: 2138 snippets, columns: ['text', 'author']
dev: 268 snippets, columns: ['text', 'author']
test: 267 snippets, columns: ['text', 'author']

Number of different authors in train: 20
Fewest snippets for one author: 51, most: 150

Snippets per author (train):
         snippets
author           
29783          91
240213         98
512464        112
560480        150
583064        130
583994         83
748687        120
806976        130
870118        141
910821         92
967934         99
1112924        93
1220273       100
1276465       121
1497577       109
2750536       132
2855986       117
2943978        51
3439302        98
6234395        71


In [14]:
#3 Fix the text (i.e. the contractions)
for df in (train, dev, test):
    df['text'] = df['text'].str.replace(r'(?<=[A-Za-z])"(?=[a-z])', "'", regex=True)

# Check that it worked
print(train['text'].iloc[0][:300])

My legs were a bit shaky, so he wrapped an arm around my waist to steady me, "Come on." There was no arguing with the Englishman, and he led me from the room as Italy tended to Germany. The hallways were filled with recovering soldiers, but for the most part the base was quiet. I wasn't used to it, 


## Part 2: Feature extraction

**Goal:** To turn each snippet into numbers that the classifier can use. At least 50 features, both lexical and syntactic organised into groups.

**What we do:**
1. Define the feature groups: lexical, character-level, function words, syntactic (POS tags) and fanfiction-specific.

    - Lexical: average word length, spread of word length, average sentence length, spread of sentence length...
    - Character-level: frequency of ecah punctuation mark, share of uppercase letters, share of digits...
    - Function words: relative frequency of common words (the, of, but...)
    - syntactic (POS tags): relative frequency of each POS tag (noun, verb, adjectiv, adverb...)

2. Write one function per group that takes the snippets and returns a table with one column per feature.
3. Apply the functions to train, dev and test so all three sets have exactly the same columns.
4. Combine the groups into one feature table per set.
5. Check the total number of features (at least 50) and the number of features per group.

**Output:** one feature table for each of 'train', 'dev' and 'test' and a list that shows which features belong to which group.

In [15]:
# 1. Define feature groups
feature_groups = {
    'lexical':        'lex_',
    'character':      'char_',
    'function_words': 'func_',
    'pos':            'pos_',
    'fanfic':         'fan_',
}

# Split snippets into words
def get_words(text):
    text = text.lower()
    tokens = text.split()
    words = []
    for token in tokens:
        word = token.strip('.,!?;:"\'()*-')
        if word:
            words.append(word)
    return words

# Split snippet into sentences
def get_sentences(text):
    parts = re.split(r'(?<=[.!?])\s+|(?<=[.!?]")\s+', text)
    sentences = []
    for part in parts:
        if part.strip():
            sentences.append(part)
    return sentences


In [16]:
# 2. Functions per group

# Lexical group
def lexical_features(texts):
    rows = []

    for text in texts:
        words = get_words(text)
        sentences = get_sentences(text)

        # Length of each word 
        word_lengths = []
        for word in words:
            word_lengths.append(len(word))
        word_lengths = np.array(word_lengths)

        # Length of each sentence
        sentence_lengths = []
        for sentence in sentences:
            sentence_lengths.append(len(get_words(sentence)))
        sentence_lengths = np.array(sentence_lengths)

        # How often each word occurs
        word_counts = Counter(words)

        # Number of words that occur once
        once_only = 0
        for count in word_counts.values():
            if count == 1:
                once_only += 1

        rows.append({
            'lex_avg_word_len': word_lengths.mean(),
            'lex_std_word_len': word_lengths.std(),
            'lex_avg_sent_len': sentence_lengths.mean(),
            'lex_std_sent_len': sentence_lengths.std(),
            'lex_ttr': len(word_counts) / len(words),
            'lex_hapax_ratio': once_only / len(words),
            'lex_long_word_share': (word_lengths >= 7).mean(),
            'lex_short_word_share': (word_lengths <= 3).mean(),
        })

    return pd.DataFrame(rows, index=texts.index)

In [17]:
# Character-level group
punct = {
    ',': 'comma',
    '.': 'period',
    ';': 'semicolon',
    ':': 'colon',
    '!': 'exclamation',
    '?': 'question',
    '-': 'hyphen',
    '(': 'parenthesis',
    '"': 'quote',
    "'": 'apostrophe',
    '*': 'asterisk',
}

def character_features(texts):
    rows = []

    for text in texts:
        n_chars = len(text)
        row = {}

        # How many times each happen
        for mark, name in punct.items():
            row['char_' + name] = text.count(mark) / n_chars

        # How many times '...' happens
        row['char_ellipsis'] = text.count('...') / n_chars

        # Uppercase letters
        n_upper = 0
        for char in text:
            if char.isupper():
                n_upper += 1
        row['char_upper_share'] = n_upper / n_chars

        # Digits
        n_digits = 0
        for char in text:
            if char.isdigit():
                n_digits += 1
        row['char_digit_share'] = n_digits / n_chars

        rows.append(row)

    return pd.DataFrame(rows, index=texts.index)

In [18]:
# Fucntion words
function_words = [
    'the', 'a', 'an', 'and', 'but', 'or', 'nor', 'so', 'yet', 'for', 'of', 'in', 'on', 'at',
    'to', 'from', 'by', 'with', 'about', 'against', 'between', 'into', 'through', 'during',
    'before', 'after', 'above', 'below', 'up', 'down', 'out', 'off', 'over', 'under', 'again',
    'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both',
    'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'not', 'only', 'own', 'same',
    'than', 'too', 'very', 'can', 'will', 'just', 'should', 'now', 'i', 'me', 'my', 'mine',
    'myself', 'we', 'us', 'our', 'you', 'your', 'he', 'him', 'his', 'she', 'her', 'hers', 'it',
    'its', 'they', 'them', 'their', 'this', 'that', 'these', 'those', 'who', 'whom', 'which',
    'what', 'whose', 'is', 'am', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has',
    'had', 'do', 'does', 'did', 'would', 'could', 'might', 'must', 'shall', 'may', 'if',
    'because', 'while', 'as', 'until', 'though', 'although', 'however', 'also', 'even',
    'still', 'almost', 'perhaps', 'maybe', 'often', 'always', 'never',
]

def function_word_features(texts):
    rows = [] 

    for text in texts:
        words = get_words(text)
        n_words = len(words)
        word_counts = Counter(words) # how many of each word
        row = {}

        # how many of each function word
        for word in function_words:
            row['func_' + word] = word_counts[word] / n_words

        rows.append(row)

    return pd.DataFrame(rows, index=texts.index)

In [19]:
# POS tagging
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner', 'lemmatizer']) # a tagger

pos_tags = ['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ'
            , 'SYM', 'VERB', 'X']

pos_bigrams = [('DET', 'NOUN'), ('ADJ', 'NOUN'), ('PRON', 'VERB'), ('NOUN', 'VERB'), ('VERB', 'ADP'), ('ADP', 'DET'), 
                ('VERB', 'ADV'), ('ADV', 'VERB'), ('PUNCT', 'PRON'), ('AUX', 'VERB')]


def pos_features(texts):
    rows = []

    for doc in nlp.pipe(texts, batch_size=50):

        # All POS tags
        tags = []
        for token in doc:
            if token.pos_ != 'SPACE':
                tags.append(token.pos_)

        n_tags = len(tags)
        tag_counts = Counter(tags)                  # how often each tag occurs
        pair_counts = Counter(zip(tags, tags[1:]))  # how often each pair of tags occurs
        row = {}

        # Individual POS tag
        for tag in pos_tags:
            row['pos_' + tag] = tag_counts[tag] / n_tags

        # Pairwise POS tag
        for first, second in pos_bigrams:
            row['pos_' + first + '_' + second] = pair_counts[(first, second)] / n_tags

        rows.append(row)

    return pd.DataFrame(rows, index=texts.index)

In [20]:
# 3. Apply functions to all data, to have the same columns
feature_tables = {}

for name, df in [('train', train), ('dev', dev), ('test', test)]:
    feature_tables[name] = {
        'lexical':        lexical_features(df['text']),
        'character':      character_features(df['text']),
        'function_words': function_word_features(df['text']),
        'pos':            pos_features(df['text']),
    }
    print(name, 'done')


for name in feature_tables:
    for group, table in feature_tables[name].items():
        print(f'{name:5} {group:15} {table.shape}')

train done
dev done
test done
train lexical         (2138, 8)
train character       (2138, 14)
train function_words  (2138, 133)
train pos             (2138, 27)
dev   lexical         (268, 8)
dev   character       (268, 14)
dev   function_words  (268, 133)
dev   pos             (268, 27)
test  lexical         (267, 8)
test  character       (267, 14)
test  function_words  (267, 133)
test  pos             (267, 27)


In [21]:
# 4. COmbine into one feature table

X_train = pd.concat(feature_tables['train'].values(), axis=1)
X_dev   = pd.concat(feature_tables['dev'].values(),   axis=1)
X_test  = pd.concat(feature_tables['test'].values(),  axis=1)

# The labels: the author of each snippet
y_train = train['author']
y_dev   = dev['author']
y_test  = test['author']

# Check: shapes, same columns in all three sets, and no missing values
print('Shapes:', X_train.shape, X_dev.shape, X_test.shape)
print('Same columns in all sets:', list(X_train.columns) == list(X_dev.columns) == list(X_test.columns))
print('Missing values in train:', X_train.isna().sum().sum())

Shapes: (2138, 182) (268, 182) (267, 182)
Same columns in all sets: True
Missing values in train: 0


In [22]:
# 5. Chek number of features and number of feature per group
print('Total number of features:', X_train.shape[1])
print()
for group, prefix in feature_groups.items():
    n_features = X_train.columns.str.startswith(prefix).sum()
    print(f'{group:15} {n_features} features')

Total number of features: 182

lexical         8 features
character       14 features
function_words  133 features
pos             27 features
fanfic          0 features


## Part 3: Classifier and tuning

**Goal:** To build a classifier that predicts the author of a snippet from its features, and tune it using the training and development sets.

**What we do:**
1. Scale the features so they are on a simialr range. The scaler is fitted on train only and then applied to dev and test.
2. Train a few candidate classifiers on the train set:

    - Logistic Regression
    - Linear SVM
    - Random Forest

3. Compare the candidates on the dev set using macro F1.
4. Tune the setings of the best classifiers on the dev set.
5. Keep the best classifier with its settings.

**Output:** the final classifier with its tuned settings and a table with the dev macro F1 of each candidate.

In [23]:
# 1. Scale feature, fit on train and appleid to dev and test
scaler = StandardScaler()
scaler.fit(X_train)

# apply the same scaling to train, dev and test 
X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns, index=X_train.index)
X_dev_scaled   = pd.DataFrame(scaler.transform(X_dev),   columns=X_dev.columns,   index=X_dev.index)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test),  columns=X_test.columns,  index=X_test.index)

print('Train: average of column means =', X_train_scaled.mean().mean().round(3))
print('Train: average of column std   =', X_train_scaled.std(ddof=0).mean().round(3))
print('Dev:   average of column means =', X_dev_scaled.mean().mean().round(3))
print('Dev:   average of column std   =', X_dev_scaled.std(ddof=0).mean().round(3))

Train: average of column means = -0.0
Train: average of column std   = 1.0
Dev:   average of column means = -0.006
Dev:   average of column std   = 0.985


## Part 4: Evaluation

**Goal:** measure how well the final classifier performs on the development set, and to check whether feature selection helps. The target is at least 0.7

**What we do:**
1. Preditc the authors of the dev snippets with the final classifier from Part 3.
2. Calculate the scores:

    - Macro F1 (the main score, because the classes are imbalanced)
    - Accuracy
    - Precision, recall and F1 for each author

3. Plot a confusion matrix to see which authors get mixed up. We need this for the failure analysis in the report.
4. Test feature selection (e.g. keeping only the k best features) and compare the dev macro F1 with and without it.
5. Decide whether feature selection is useful in our case.

**Output:** the dev scores (overall and per author), the confusion matrix and a comparison of the results with and without feature selection.

## Part 5: Ablation analysis

**Goal:** find out which feature group is the most informative for the classifier by removing one group at a time and checking how the performance changes.

**What we do:**
1. Train the final classifier with all feature groups and record the dev macro F1. This is the baseline (no groups removed).
2. For each feature group:

    - Remove all features of that group
    - Retrain the classifier with the same settings
    - Record the dev macro F1

3. Plot the results as a bar chart in the style of Figure 1 of the assignment: one bar for the baseline and one bar for each removed group.
4. Compare the bars: the group dropping the most in F1 score is the most informative.

**Output:** a bar chart saved in the results folder and a table with the dev macro F1 for each removed group.

## Part 6: Test set

**Goal:** run the final classifier on the unseen test set and see how well it generalises. The test set is used once and nothing is changed after seeing the result.

**What we do:**
1. Take the final classifier with its settings and the selected features.
2. Predict the authors of the test snippets.
3. Calculate the same scores as for the dev set:

    - Macro F1 and accuracy
    - Precision, recall and F1 for each author

4. Plot the confusion matrix for the test set.
5. Compare the test scores with the dev scores and note any difference.

**Output:** the test scores (overall and per author), the confusion matrix and a comparison of dev and test performance.